# FIGNN — Research Validation & Robustness Notebook

This notebook is for **research validation**, not presentation.

It consolidates the completed Stage-1/Stage-2 analyses and adds the targeted checks motivated by the review corrections:

1. Audit the actual training/validation split.
2. Preserve silhouette as a **development diagnostic**, not the primary transferability metric.
3. Recheck the completed 29-subject trajectory geometry.
4. Test the 3-point Procrustes result with **1,000 random workload-label alterations per subject**.
5. Examine the REST anchor using four-condition latent-space visualization and REST-to-N-back geometry.
6. Consolidate Stage-2 error statistics and paired inference.
7. Search for historical smaller-architecture checkpoints before deciding whether a new 8-head ablation must be trained.

**No model training is performed by default.** Frozen-encoder embedding extraction is performed only for the new REST/Procrustes analyses.


In [ ]:
# ============================================================
# 0. SETUP
# ============================================================

import os
import sys
import re
import glob
import json
import copy
import pickle
import warnings

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
import plotly.express as px

from scipy.spatial import procrustes
from scipy.stats import wilcoxon, ttest_rel
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from torch_geometric.loader import DataLoader

warnings.filterwarnings("ignore")

PROJECT_ROOT = "/home/nvidia/24BCE1793/fignn_cni__project"
GRAPH_PATH = "/lp-dev/24BCE1793/data/processed/all_graphs_29sub.pkl"
LOSO_ROOT = "/lp-dev/24BCE1793/outputs/loso"
STAGE1_ROOT = "/lp-dev/24BCE1793/outputs/evaluation_loso_all29"
STAGE2_CAL_PATH = "/lp-dev/24BCE1793/outputs/stage2_calibration_comparison_64D.csv"
STAGE2_DIST_PATH = "/lp-dev/24BCE1793/outputs/stage2_centroid_64D_distances.csv"

SUBJECTS = list(range(1, 30))
TASK_LABELS = [0, 1, 2]
REST_LABEL = 3
LABEL_NAMES = {0: "0-back", 1: "1-back", 2: "2-back", 3: "REST"}

N_PERM = 1000
RNG_SEED = 20260906

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

sys.path.insert(0, PROJECT_ROOT)
from src.models.model import FIGNNEncoder

print("Project:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Permutation draws per subject:", N_PERM)


In [ ]:
# ============================================================
# 1. HELPERS
# ============================================================

def get_subject(g):
    for name in ["subject", "subj", "subject_id", "sid"]:
        if hasattr(g, name):
            value = getattr(g, name)
            return int(value.item()) if torch.is_tensor(value) else int(value)
    raise AttributeError("Could not find subject ID.")

def get_label(g):
    return int(g.y.item())

def compute_train_stats(graphs):
    all_x = torch.cat([g.x for g in graphs], dim=0)
    mean = all_x.mean(dim=0)
    std = all_x.std(dim=0).clamp(min=1e-8)
    return mean, std

def normalize_graphs(graphs, mean, std):
    out = []
    for g in graphs:
        h = copy.copy(g)
        h.x = (g.x.clone() - mean) / std
        out.append(h)
    return out

@torch.no_grad()
def encode_graphs(model, graphs, mean, std, batch_size=128):
    if not graphs:
        return np.empty((0, 64)), np.empty((0,), dtype=int)
    loader = DataLoader(
        normalize_graphs(graphs, mean, std),
        batch_size=batch_size,
        shuffle=False,
        num_workers=0
    )
    zs, ys = [], []
    model.eval()
    for batch in loader:
        batch = batch.to(DEVICE)
        z = model(batch)
        zs.append(z.cpu().numpy())
        ys.append(batch.y.view(-1).cpu().numpy())
    return np.vstack(zs), np.concatenate(ys)

def centroid(z, y, label):
    return z[y == label].mean(axis=0)

def trajectory(z, y):
    return np.vstack([centroid(z, y, l) for l in TASK_LABELS])

def trajectory_geometry(T):
    d01 = np.linalg.norm(T[1] - T[0])
    d12 = np.linalg.norm(T[2] - T[1])
    d02 = np.linalg.norm(T[2] - T[0])
    a = T[0] - T[1]
    b = T[2] - T[1]
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    angle = (
        np.degrees(np.arccos(np.clip(np.dot(a, b) / denom, -1, 1)))
        if denom > 1e-12 else np.nan
    )
    efficiency = d02 / (d01 + d12) if (d01 + d12) > 1e-12 else np.nan
    return d01, d12, d02, angle, efficiency

def load_fold_model(subject, checkpoint_name="fignn_best.pth"):
    ckpt = os.path.join(LOSO_ROOT, f"subject_{subject:02d}", checkpoint_name)
    if not os.path.exists(ckpt):
        raise FileNotFoundError(ckpt)
    model = FIGNNEncoder().to(DEVICE)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()
    return model

def load_fold_stats(subject, training_graphs):
    path = os.path.join(
        LOSO_ROOT, f"subject_{subject:02d}", "normalisation_stats.pt"
    )
    if os.path.exists(path):
        s = torch.load(path, map_location="cpu")
        return s["mean"].float(), s["std"].float().clamp(min=1e-8)
    return compute_train_stats(training_graphs)

def bh_fdr(pvals):
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    ranked = pvals[order]
    q = ranked * len(pvals) / np.arange(1, len(pvals) + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty_like(q)
    out[order] = q
    return np.minimum(out, 1.0)


## 2. Training / validation audit

The current `train.py` computes normalization statistics from the supplied training graph set, then creates validation data using a **random graph-level permutation**. Thus the held-out LOSO subject is protected, but graphs from the same training subjects can appear in both train and validation.

This is a **code-audit finding**, not a new experiment. The uploaded training code shows the random `torch.randperm` split and silhouette-based checkpoint selection. fileciteturn117file0L151-L158 fileciteturn117file0L221-L237


In [ ]:
# ============================================================
# 2. TRAINING / VALIDATION AUDIT
# ============================================================

TRAIN_PY = os.path.join(PROJECT_ROOT, "train.py")

if os.path.exists(TRAIN_PY):
    train_source = open(TRAIN_PY, "r", encoding="utf-8").read()
else:
    train_source = None

audit = {
    "normalization": "z-score on node features (4 relative-band-power features)",
    "normalization_scope": "train_graphs supplied to run_training()",
    "validation_split": "random graph-level split using torch.randperm",
    "validation_subject_isolation": "not guaranteed within the 28 training subjects",
    "held_out_LOSO_subject": "excluded from train_graphs in the LOSO pipeline",
    "checkpoint_selection": "highest validation silhouette",
}

for k, v in audit.items():
    print(f"{k}: {v}")

if train_source is not None:
    print("\nSource check:")
    for needle in ["torch.randperm", "silhouette_score", "best_sil", "compute_dataset_stats"]:
        print(f"  {needle}: {'FOUND' if needle in train_source else 'NOT FOUND'}")
else:
    print("\ntrain.py not found at project root; audit above is based on the uploaded train.py.")


## 3. Silhouette: retain as a development diagnostic

Silhouette remains in the research record, but its role is explicit:

- it was used during development/checkpoint selection;
- weak or negative values indicate that the latent representation was not simply forming three compact, isolated clusters;
- this motivated examining workload as a **structured latent trajectory** rather than requiring conventional cluster separation;
- it is secondary evidence, not the primary transferability metric.

The current implementation selects the best checkpoint by validation silhouette. fileciteturn117file0L221-L245


In [ ]:
# ============================================================
# 3. SILHOUETTE HISTORY AUDIT
# ============================================================

log_files = sorted(glob.glob(os.path.join(LOSO_ROOT, "subject_*.log")))
sil_rows = []

for path in log_files:
    m = re.search(r"subject_(\d+)\.log$", os.path.basename(path))
    if not m:
        continue
    sid = int(m.group(1))
    text = open(path, "r", errors="ignore").read()
    vals = [float(x) for x in re.findall(r"val_silhouette=([-+]?\d*\.\d+)", text)]
    if vals:
        sil_rows.append({
            "subject": sid,
            "n_checks": len(vals),
            "min_val_silhouette": min(vals),
            "max_val_silhouette": max(vals),
            "final_recorded_silhouette": vals[-1],
            "negative_checks": sum(v < 0 for v in vals),
        })

sil_history_df = pd.DataFrame(sil_rows).sort_values("subject")

if len(sil_history_df):
    display(sil_history_df.round(4))
    print("\nNegative validation-silhouette observations:",
          int(sil_history_df["negative_checks"].sum()))
    print("Subjects with at least one negative validation silhouette:",
          int((sil_history_df["negative_checks"] > 0).sum()))
else:
    print("No LOSO logs found; no historical silhouette values can be reconstructed here.")


## 4. Completed Stage-1 geometry — verification only

The distance/trajectory analysis is **already completed**. This section loads the authoritative all-29 LOSO results and does not introduce a new monotonicity test.

Existing analyses already computed \(d_{01}\), \(d_{12}\), \(d_{02}\), trajectory angle, path efficiency, and the 28/29 ordering result. fileciteturn118file5L603-L609


In [ ]:
# ============================================================
# 4. AUTHORITATIVE STAGE-1 RESULTS
# ============================================================

stage1_csv = os.path.join(STAGE1_ROOT, "loso_all29_subject_metrics.csv")
stage1_json = os.path.join(STAGE1_ROOT, "loso_all29_report.json")

stage1_df = pd.read_csv(stage1_csv)
with open(stage1_json, "r") as f:
    stage1_report = json.load(f)

print("Subjects:", stage1_report["n_subjects"])
display(stage1_df.round(4))

metrics = stage1_report["metrics"]
counts = stage1_report["counts"]

print("\nMean Procrustes similarity:",
      metrics["procrustes_similarity"]["mean"])
print("0B-1B < 0B-2B:",
      counts["monotonic_d01_lt_d02"], "/", stage1_report["n_subjects"])
print("1B between 0B and 2B:",
      counts["oneback_between"], "/", stage1_report["n_subjects"])


## 5. Procrustes permutation null — primary new robustness test

For every held-out subject:

1. keep the subject's three latent centroids unchanged;
2. randomly permute their workload labels;
3. compute Procrustes similarity against the fold-specific population trajectory;
4. repeat **1,000 times**;
5. compare the observed similarity with the null distribution.

The correction specifically recommends 1,000 random label permutations per subject to test whether the observed similarity is achievable by chance. fileciteturn116file0L22-L33

The notebook uses the fold-specific `fignn_best.pth` checkpoint and training-only normalization statistics so the null is evaluated in the same frozen LOSO setting as the structured trajectory analysis.


In [ ]:
# ============================================================
# 5. PROCRUSTES PERMUTATION NULL
# ============================================================

all_graphs = None
trajectory_cache = {}
population_cache = {}

def load_graphs_once():
    global all_graphs
    if all_graphs is None:
        print("Loading graphs...")
        with open(GRAPH_PATH, "rb") as f:
            all_graphs = pickle.load(f)
        print("Graphs:", len(all_graphs))
    return all_graphs

def get_fold_trajectories(subject):
    """Return population and held-out 3x64 trajectories for one LOSO fold."""
    if subject in trajectory_cache:
        return population_cache[subject], trajectory_cache[subject]

    graphs = load_graphs_once()

    train_graphs = [
        g for g in graphs
        if get_subject(g) != subject and get_label(g) in TASK_LABELS
    ]
    test_graphs = [
        g for g in graphs
        if get_subject(g) == subject and get_label(g) in TASK_LABELS
    ]

    model = load_fold_model(subject, "fignn_best.pth")
    mean, std = load_fold_stats(subject, train_graphs)

    z_train, y_train = encode_graphs(model, train_graphs, mean, std)
    z_test, y_test = encode_graphs(model, test_graphs, mean, std)

    training_subjects = sorted(set(get_subject(g) for g in train_graphs))

    pop = []
    for label in TASK_LABELS:
        per_subject = []
        for sid in training_subjects:
            mask = np.array([
                get_subject(g) == sid and get_label(g) == label
                for g in train_graphs
            ])
            per_subject.append(z_train[mask].mean(axis=0))
        pop.append(np.mean(per_subject, axis=0))

    pop = np.vstack(pop)
    subj = trajectory(z_test, y_test)

    population_cache[subject] = pop
    trajectory_cache[subject] = subj

    del model, z_train, z_test
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return pop, subj

rng = np.random.default_rng(RNG_SEED)
perm_rows = []

for _, row in stage1_df.sort_values("subject").iterrows():
    sid = int(row["subject"])

    pop_T, subj_T = get_fold_trajectories(sid)

    observed_sim = 1.0 - procrustes(pop_T, subj_T)[2]

    null_sims = np.empty(N_PERM)
    for j in range(N_PERM):
        perm = rng.permutation(3)
        null_sims[j] = 1.0 - procrustes(pop_T, subj_T[perm])[2]

    ge = int(np.sum(null_sims >= observed_sim))
    p_raw = ge / N_PERM
    p_plus1 = (ge + 1) / (N_PERM + 1)
    percentile = 100.0 * np.mean(null_sims < observed_sim)

    perm_rows.append({
        "subject": sid,
        "observed_procrustes": observed_sim,
        "null_mean": null_sims.mean(),
        "null_sd": null_sims.std(ddof=1),
        "null_q025": np.quantile(null_sims, 0.025),
        "null_q975": np.quantile(null_sims, 0.975),
        "null_ge_observed": ge,
        "empirical_p_raw": p_raw,
        "empirical_p_plus1": p_plus1,
        "observed_percentile": percentile,
    })

perm_df = pd.DataFrame(perm_rows).sort_values("subject").reset_index(drop=True)
perm_df["q_fdr"] = bh_fdr(perm_df["empirical_p_plus1"].values)

display(perm_df.round(5))

print("\nSubjects with p < 0.05:",
      int((perm_df["empirical_p_plus1"] < 0.05).sum()), "/", len(perm_df))
print("Subjects with FDR q < 0.05:",
      int((perm_df["q_fdr"] < 0.05).sum()), "/", len(perm_df))
print("Median empirical p:",
      float(perm_df["empirical_p_plus1"].median()))
print("Median observed percentile:",
      float(perm_df["observed_percentile"].median()))


In [ ]:
# ============================================================
# 5A. PROCRUSTES NULL SUMMARY PLOT
# ============================================================

fig = px.scatter(
    perm_df,
    x="subject",
    y="observed_percentile",
    hover_data=["empirical_p_plus1", "observed_procrustes"],
    title="Observed Procrustes Similarity: Percentile Within 1,000-Draw Null"
)

fig.add_hline(y=95, line_dash="dash")
fig.update_layout(
    template="plotly_white",
    xaxis_title="Held-out subject",
    yaxis_title="Observed percentile in permutation null",
    height=500
)
fig.show()


### Permutation note

There are only six unique permutations of three labels, so 1,000 random draws necessarily revisit those six orderings. We nevertheless use **1,000 random alterations exactly as requested by the correction**, with a fixed seed for reproducibility.

The purpose is to obtain the requested empirical null under random workload-label assignment, not to imply that there are 1,000 unique labelings.


## 6. REST-anchor validation

This is a new Stage-2-supporting analysis.

Question:

> Does the frozen encoder place REST in a geometrically distinct region relative to the N-back states, making it a plausible subject-specific reference point?

For a representative held-out subject, the notebook visualizes **REST + 0-back + 1-back + 2-back** in a joint PCA projection. Quantitative distances remain in the original 64-D latent space.

The four-class silhouette is retained only as a **descriptive cluster diagnostic**.


In [ ]:
# ============================================================
# 6. REST ANCHOR — REPRESENTATIVE SUBJECT
# ============================================================

REST_SUBJECT = 17

graphs = load_graphs_once()
fold_train = [g for g in graphs if get_subject(g) != REST_SUBJECT]
fold_test = [g for g in graphs if get_subject(g) == REST_SUBJECT]

workload_train = [g for g in fold_train if get_label(g) in TASK_LABELS]

model = load_fold_model(REST_SUBJECT, "fignn_best.pth")
mean, std = load_fold_stats(REST_SUBJECT, workload_train)

z_rest_demo, y_rest_demo = encode_graphs(model, fold_test, mean, std)

four_labels = [REST_LABEL] + TASK_LABELS
four_centroids = np.vstack([
    centroid(z_rest_demo, y_rest_demo, label)
    for label in four_labels
])

pca4 = PCA(n_components=2)
four_2d = pca4.fit_transform(np.vstack([z_rest_demo, four_centroids]))

sample_2d = four_2d[:-4]
cent_2d = four_2d[-4:]

plot_df = pd.DataFrame({
    "PC1": sample_2d[:, 0],
    "PC2": sample_2d[:, 1],
    "Condition": [LABEL_NAMES[int(y)] for y in y_rest_demo]
})

fig = px.scatter(
    plot_df,
    x="PC1",
    y="PC2",
    color="Condition",
    opacity=0.35,
    title=f"REST and N-back Latent Geometry — Held-out Subject {REST_SUBJECT}"
)

fig.add_trace(go.Scatter(
    x=cent_2d[:, 0],
    y=cent_2d[:, 1],
    mode="markers+text",
    text=[LABEL_NAMES[l] for l in four_labels],
    textposition="top center",
    marker=dict(size=13, symbol="diamond"),
    name="Centroids"
))

fig.update_layout(template="plotly_white", height=600)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.show()

try:
    sil4 = silhouette_score(z_rest_demo, y_rest_demo)
    print(f"Four-class silhouette (REST + 0/1/2): {sil4:.6f}")
    print("Descriptive only; not a primary trajectory metric.")
except Exception as e:
    print("Could not compute four-class silhouette:", e)

rest_c = centroid(z_rest_demo, y_rest_demo, REST_LABEL)
task_cs = {l: centroid(z_rest_demo, y_rest_demo, l) for l in TASK_LABELS}

rest_dist = {
    f"REST_to_{LABEL_NAMES[l]}": float(np.linalg.norm(rest_c - task_cs[l]))
    for l in TASK_LABELS
}

print("\n64-D REST anchor distances:")
for k, v in rest_dist.items():
    print(f"  {k}: {v:.6f}")


In [ ]:
# ============================================================
# 6A. REST ANCHOR — ALL 29 SUBJECTS
# ============================================================

rest_rows = []

for sid in SUBJECTS:
    graphs = load_graphs_once()

    train_g = [
        g for g in graphs
        if get_subject(g) != sid and get_label(g) in TASK_LABELS
    ]
    test_g = [g for g in graphs if get_subject(g) == sid]

    model = load_fold_model(sid, "fignn_best.pth")
    mean, std = load_fold_stats(sid, train_g)
    z, y = encode_graphs(model, test_g, mean, std)

    if not all(np.any(y == l) for l in [REST_LABEL, 0, 1, 2]):
        print(f"Subject {sid}: missing REST/0/1/2; skipped.")
        continue

    cR = centroid(z, y, REST_LABEL)
    c0 = centroid(z, y, 0)
    c1 = centroid(z, y, 1)
    c2 = centroid(z, y, 2)

    dR0 = np.linalg.norm(cR - c0)
    dR1 = np.linalg.norm(cR - c1)
    dR2 = np.linalg.norm(cR - c2)

    rest_rows.append({
        "subject": sid,
        "d_REST_0B": dR0,
        "d_REST_1B": dR1,
        "d_REST_2B": dR2,
        "d_REST_1B_over_0B": dR1 / dR0 if dR0 > 1e-12 else np.nan,
        "d_REST_2B_over_0B": dR2 / dR0 if dR0 > 1e-12 else np.nan,
    })

    del model, z
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

rest_df = pd.DataFrame(rest_rows)

display(rest_df.round(5))

print("\nMean REST distances:")
display(
    rest_df[["d_REST_0B", "d_REST_1B", "d_REST_2B"]]
    .agg(["mean", "median", "std"])
    .round(5)
)


## 7. Existing trajectory geometry — consolidated, not new

No new monotonicity claim is introduced here. The completed analysis already contains the trajectory distances, ratios, angle, path efficiency, and 28/29 ordering result. fileciteturn118file5L603-L609


In [ ]:
# ============================================================
# 7. EXISTING GEOMETRY — CONSOLIDATED
# ============================================================

geometry_cols = [
    c for c in [
        "subject", "d01", "d12", "d02",
        "d01_d02_ratio", "d12_d02_ratio",
        "angle_deg", "path_efficiency",
        "monotonicity", "oneback_between"
    ]
    if c in stage1_df.columns
]

if len(geometry_cols) > 1:
    display(stage1_df[geometry_cols].sort_values("subject").round(5))

print("\nExisting completed summary:")
print("Mean d01/d02 = 0.3657")
print("Mean d12/d02 = 0.7692")
print("Mean trajectory angle = 53.39 deg")
print("Mean path efficiency = 0.9015")


## 8. Stage-2 statistical evidence

The final Stage-2 comparison is already defined in the original 64-D latent space.

This section reports:

- mean ± SD
- median
- absolute reduction
- relative reduction
- 22/29 improved
- one-sided paired Wilcoxon signed-rank test
- paired t-test as a sensitivity analysis

The Wilcoxon test is the primary inferential test.


In [ ]:
# ============================================================
# 8. STAGE 2 STATISTICS
# ============================================================

cal_df = pd.read_csv(STAGE2_CAL_PATH)

baseline = cal_df["baseline_error_64D"].astype(float)
calibrated = cal_df["calibrated_error_64D"].astype(float)
diff = baseline - calibrated

w_stat, w_p = wilcoxon(
    baseline,
    calibrated,
    alternative="greater"
)

t_stat, t_p = ttest_rel(baseline, calibrated)

mean_base = baseline.mean()
mean_cal = calibrated.mean()
abs_reduction = mean_base - mean_cal
relative_reduction = 100 * abs_reduction / mean_base

stage2_stats = pd.DataFrame([{
    "N": len(cal_df),
    "Baseline mean": mean_base,
    "Baseline SD": baseline.std(ddof=1),
    "Baseline median": baseline.median(),
    "Calibrated mean": mean_cal,
    "Calibrated SD": calibrated.std(ddof=1),
    "Calibrated median": calibrated.median(),
    "Absolute mean reduction": abs_reduction,
    "Relative mean reduction (%)": relative_reduction,
    "Subjects improved": int((diff > 0).sum()),
    "Wilcoxon statistic": w_stat,
    "Wilcoxon one-sided p": w_p,
    "Paired t statistic": t_stat,
    "Paired t p": t_p,
}])

display(stage2_stats.round(8))
print("\nWilcoxon H1: baseline error > calibrated error.")


In [ ]:
# ============================================================
# 8A. STAGE 2 EFFECT SIZE
# ============================================================

nonzero = diff[diff != 0].to_numpy()
ranks = pd.Series(np.abs(nonzero)).rank(method="average").to_numpy()
W_plus = ranks[nonzero > 0].sum()
W_minus = ranks[nonzero < 0].sum()
rank_biserial = (W_plus - W_minus) / (W_plus + W_minus)

print(f"Wilcoxon rank-biserial correlation: {rank_biserial:.6f}")

fig = px.box(
    pd.DataFrame({
        "Method": (
            ["Population-only"] * len(cal_df)
            + ["REST + 2-back calibrated"] * len(cal_df)
        ),
        "64-D error": np.r_[baseline, calibrated]
    }),
    x="Method",
    y="64-D error",
    points="all",
    title="Stage 2 — 64-D Estimation Error"
)

fig.update_layout(template="plotly_white", height=500)
fig.show()


## 9. Architecture ablation — investigate before retraining

The final encoder uses hidden dimension 128, eight first-layer heads, and latent dimension 64. Because the first layer concatenates the eight heads, its intermediate representation is \(8\times128=1024\).

The correction asks whether this is over-parameterized. A true ablation requires **retraining controlled alternative architectures**; changing an already-trained checkpoint is not an ablation.

The current model source records the earlier configuration as 64 hidden / 4 heads / 32 latent and the final configuration as 128 hidden / 8 heads / 64 latent. fileciteturn120file0L21-L28


In [ ]:
# ============================================================
# 9. HISTORICAL ARCHITECTURE SEARCH
# ============================================================

patterns = [
    r"hidden_dim\s*[:=]\s*64",
    r"latent_dim\s*[:=]\s*32",
    r"n_heads\s*[:=]\s*4",
]

search_roots = [
    os.path.join(PROJECT_ROOT, "outputs"),
    os.path.join(PROJECT_ROOT, "logs"),
    os.path.join(PROJECT_ROOT, "checkpoints"),
]

candidate_files = []
for root in search_roots:
    if os.path.exists(root):
        candidate_files.extend(glob.glob(os.path.join(root, "**", "*"), recursive=True))

matches = []
for path in candidate_files:
    if not os.path.isfile(path) or os.path.getsize(path) > 5_000_000:
        continue
    try:
        text = open(path, "r", errors="ignore").read()
    except Exception:
        continue
    hit = [p for p in patterns if re.search(p, text, flags=re.I)]
    if hit:
        matches.append((path, hit))

print(f"Files with historical smaller-architecture references: {len(matches)}")
for path, hit in matches[:30]:
    print(" ", path, "->", hit)

print("\nNo new ablation training is launched by this notebook.")


### Formal ablation decision

If historical smaller checkpoints/results exist, evaluate those first.

If they do not exist, the clean formal comparison is:

| Configuration | hidden | heads | latent |
|---|---:|---:|---:|
| Smaller | 64 | 4 | 32 |
| Final | 128 | 8 | 64 |

with identical preprocessing, SupCon temperature, optimizer, training duration, LOSO split, and checkpoint-selection protocol.

**Do not claim an ablation result until the alternative model has actually been retrained and evaluated.**


In [ ]:
# ============================================================
# 10. FINAL RESEARCH SUMMARY
# ============================================================

print("=" * 78)
print("FIGNN RESEARCH VALIDATION SUMMARY")
print("=" * 78)

print("\nPRIMARY STAGE 1")
print(f"  Mean Procrustes similarity: {stage1_report['metrics']['procrustes_similarity']['mean']:.3f}")
print(f"  d01 < d02: {stage1_report['counts']['monotonic_d01_lt_d02']}/29")

if "perm_df" in globals():
    print("\nPROCRUSTES NULL")
    print(f"  Random alterations per subject: {N_PERM}")
    print(f"  Subjects p < .05: {(perm_df['empirical_p_plus1'] < 0.05).sum()}/29")
    print(f"  Subjects FDR q < .05: {(perm_df['q_fdr'] < 0.05).sum()}/29")
    print(f"  Median observed percentile: {perm_df['observed_percentile'].median():.2f}")

print("\nREST ANCHOR")
if "rest_df" in globals() and len(rest_df):
    print(f"  Subjects computed: {len(rest_df)}/29")
    print(f"  Mean REST→0B: {rest_df['d_REST_0B'].mean():.5f}")
    print(f"  Mean REST→1B: {rest_df['d_REST_1B'].mean():.5f}")
    print(f"  Mean REST→2B: {rest_df['d_REST_2B'].mean():.5f}")

print("\nSTAGE 2")
print(f"  Mean baseline error: {mean_base:.6f}")
print(f"  Mean calibrated error: {mean_cal:.6f}")
print(f"  Relative mean reduction: {relative_reduction:.2f}%")
print(f"  Subjects improved: {(diff > 0).sum()}/29")
print(f"  Wilcoxon one-sided p: {w_p:.8g}")

print("\nSECONDARY / DEVELOPMENT")
print("  Direction cosine: secondary supporting diagnostic.")
print("  Silhouette: retained as development/secondary diagnostic.")
print("  Distance ratios / monotonicity: already completed.")
print("  Architecture ablation: pending unless historical smaller-model evidence is found.")
print("  Validation split: random graph-level split within the 28 training subjects; manuscript wording should reflect this.")
